In [1]:
import requests
import pandas as pd
import time
import os

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
END_DATE = "2022-12-31" # 3 Yıl (36 Ay)

# Çıktı Dosya İsimleri
FILE_UV = "Rapor_Aylik_UV_Index.xlsx"
FILE_SUN = "Rapor_Aylik_Guneslenme_Suresi.xlsx"

# Şehir Listesi (Senin formatın: Kuzey/Batı/Güney/Doğu)
# Open-Meteo nokta bazlı çalıştığı için bu kutunun tam ortasını alacağız.
# Şehir Listesi (Verdiğiniz Sıralı Liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    """
    Verilen 'LatMax/LonMin/LatMin/LonMax' stringini parçalar 
    ve orta noktasını (Centroid Lat, Centroid Lon) döndürür.
    """
    try:
        parts = [float(x) for x in coord_string.split('/')]
        # Eğer 4 parça varsa (Kutu formatı)
        if len(parts) == 4:
            lat1, lon1, lat2, lon2 = parts
            center_lat = (lat1 + lat2) / 2
            center_lon = (lon1 + lon2) / 2
            return center_lat, center_lon
        # Eğer sadece 2 parça varsa (Direkt nokta)
        elif len(parts) == 2:
            return parts[0], parts[1]
    except:
        return None, None
    return None, None

def fetch_open_meteo_data(city_name, lat, lon):
    """
    Open-Meteo Arşiv API'sinden 3 yıllık günlük veriyi çeker.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["uv_index_max", "sunshine_duration"], # İstenen metrikler
        "timezone": "auto"
    }
    
    print(f"📡 Veri çekiliyor: {city_name} ({lat:.2f}, {lon:.2f})...")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Hata varsa fırlat
        return response.json()
    except Exception as e:
        print(f"❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []
sun_data_list = []

print(f"⏳ İşlem Başlıyor... ({START_DATE} - {END_DATE})\n")

for city, coords in LOCATIONS.items():
    # 1. Koordinatları Hesapla
    lat, lon = parse_coordinates_to_centroid(coords)
    
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue

    # 2. Veriyi Çek
    data = fetch_open_meteo_data(city, lat, lon)
    
    if data and "daily" in data:
        # 3. Pandas DataFrame'e çevir
        df = pd.DataFrame({
            "date": pd.to_datetime(data["daily"]["time"]),
            "uv_max": data["daily"]["uv_index_max"],
            "sun_sec": data["daily"]["sunshine_duration"]
        })
        
        # 4. Veri Temizliği ve Dönüşüm
        # Güneşlenme süresi saniyeden saate çevrilir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # 5. Aylık Ortalamaları Al (Resample)
        # 'ME' = Month End (Her ayın sonuna göre grupla)
        monthly = df.resample('ME', on='date').mean()
        
        # 6. Satır verilerini hazırla (Sütunlar: 2020-01, 2020-02...)
        # UV Satırı
        uv_row = {"Sehir": city}
        # Sunshine Satırı
        sun_row = {"Sehir": city}
        
        for date_idx, row in monthly.iterrows():
            col_name = date_idx.strftime("%Y-%m") # Örn: 2020-01
            
            # UV: 2 ondalık basamak
            uv_row[col_name] = round(row["uv_max"], 2) if pd.notnull(row["uv_max"]) else 0
            
            # Sun: 2 ondalık basamak (Günlük Ortalama Saat)
            sun_row[col_name] = round(row["sun_hours"], 2) if pd.notnull(row["sun_hours"]) else 0
            
        uv_data_list.append(uv_row)
        sun_data_list.append(sun_row)
        
    else:
        print(f"⚠️ Veri boş döndü: {city}")

    # API nezaketi (Hızlı istekleri engellememesi için)
    time.sleep(1)

# --- EXCEL OLUŞTURMA ---

print("\n💾 Excel dosyaları oluşturuluyor...")

# 1. UV Index Dosyası
if uv_data_list:
    df_uv_final = pd.DataFrame(uv_data_list)
    df_uv_final.to_excel(FILE_UV, index=False)
    print(f"✅ {FILE_UV} başarıyla oluşturuldu! ({len(uv_data_list)} şehir)")

# 2. Sunshine Duration Dosyası
if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    print(f"✅ {FILE_SUN} başarıyla oluşturuldu! ({len(sun_data_list)} şehir)")

print("\n🏁 İşlem Tamamlandı.")

⏳ İşlem Başlıyor... (2020-01-01 - 2022-12-31)

📡 Veri çekiliyor: Birmingham - Alabama (33.52, -86.81)...
📡 Veri çekiliyor: Anchorage - Alaska (61.17, -149.92)...
📡 Veri çekiliyor: Ankara - Turkiye (39.90, 32.75)...
📡 Veri çekiliyor: London - UK (51.50, -0.05)...

💾 Excel dosyaları oluşturuluyor...
✅ Rapor_Aylik_UV_Index.xlsx başarıyla oluşturuldu! (4 şehir)
✅ Rapor_Aylik_Guneslenme_Suresi.xlsx başarıyla oluşturuldu! (4 şehir)

🏁 İşlem Tamamlandı.


In [2]:
import requests
import pandas as pd
import time
import os

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
END_DATE = "2022-12-31" # 2020, 2021, 2022 Tamamı

# Çıktı Dosya İsimleri
FILE_UV = "Rapor_Aylik_Ortalama_UV_Index.xlsx"
FILE_SUN = "Rapor_Aylik_Toplam_Guneslenme_Saati.xlsx"

# Şehir Listesi (Verdiğiniz Sıralı Liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    """
    Verilen 'LatMax/LonMin/LatMin/LonMax' stringini parçalar 
    ve orta noktasını (Centroid Lat, Centroid Lon) döndürür.
    """
    try:
        parts = [float(x) for x in coord_string.split('/')]
        # LatMax, LonMin, LatMin, LonMax -> ERA5 formatı
        # Open-Meteo için orta nokta alıyoruz.
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["uv_index_max", "sunshine_duration"], 
        "timezone": "auto"
    }
    
    # Bilgi mesajı
    print(f"📡 Veri çekiliyor ({city_name})...", end="\r")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []
sun_data_list = []

print(f"⏳ İşlem Başlıyor... ({START_DATE} - {END_DATE})")
print(f"🌍 Toplam Şehir Sayısı: {len(ERA5_CITY_AREAS)}\n")

for city, coords in ERA5_CITY_AREAS.items():
    # 1. Koordinatları Hesapla (Orta Nokta)
    lat, lon = parse_coordinates_to_centroid(coords)
    
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue

    # 2. Veriyi Çek
    data = fetch_open_meteo_data(city, lat, lon)
    
    if data and "daily" in data:
        df = pd.DataFrame({
            "date": pd.to_datetime(data["daily"]["time"]),
            "uv_max": data["daily"]["uv_index_max"],
            "sun_sec": data["daily"]["sunshine_duration"]
        })
        
        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # Veri eksiklerini 0 ile doldur (özellikle gece/kutup bölgeleri için)
        df = df.fillna(0)

        # --- AYLIK HESAPLAMALAR ---
        # 1. UV: Aylık ORTALAMA (Mean)
        # 2. Güneşlenme: Aylık TOPLAM (Sum) -> Bu, "Günlük Ort x Gün Sayısı" ile aynıdır.
        
        monthly_uv = df.resample('ME', on='date')["uv_max"].mean()
        monthly_sun = df.resample('ME', on='date')["sun_hours"].sum()
        
        # Satır verilerini hazırla
        uv_row = {"Sehir": city}
        sun_row = {"Sehir": city}
        
        for date_idx, val in monthly_uv.items():
            col_name = date_idx.strftime("%Y-%m")
            uv_row[col_name] = round(val, 2)
            
        for date_idx, val in monthly_sun.items():
            col_name = date_idx.strftime("%Y-%m")
            sun_row[col_name] = round(val, 2) # Toplam Saat
            
        uv_data_list.append(uv_row)
        sun_data_list.append(sun_row)
        
    else:
        print(f"\n⚠️ Veri boş döndü: {city}")

    # API nezaketi
    time.sleep(0.5)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

# --- EXCEL OLUŞTURMA ---

# 1. UV Index Dosyası (Ortalama)
if uv_data_list:
    df_uv_final = pd.DataFrame(uv_data_list)
    df_uv_final.to_excel(FILE_UV, index=False)
    print(f"✅ UV Dosyası Hazır: {FILE_UV}")

# 2. Sunshine Duration Dosyası (Aylık Toplam)
if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    print(f"✅ Güneşlenme Dosyası Hazır: {FILE_SUN}")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ İşlem Başlıyor... (2020-01-01 - 2022-12-31)
🌍 Toplam Şehir Sayısı: 119



/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


...

/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


/var/folders/qb/ds_3nf0j0yl501mlfq8r3v1c0000gn/T/ipykernel_14736/4276802360.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


KeyboardInterrupt: 

In [3]:
import requests
import pandas as pd
import time
import os

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
END_DATE = "2022-12-31" # 2020, 2021, 2022 Tamamı

# Çıktı Dosya İsimleri
FILE_UV = "Rapor_Aylik_Ortalama_UV_Index.xlsx"
FILE_SUN = "Rapor_Aylik_Toplam_Guneslenme_Saati.xlsx"

# Şehir Listesi
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["uv_index_max", "sunshine_duration"], 
        "timezone": "auto"
    }
    print(f"📡 Veri çekiliyor ({city_name})...", end="\r")
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []
sun_data_list = []

print(f"⏳ İşlem Başlıyor... ({START_DATE} - {END_DATE})")
print(f"🌍 Toplam Şehir Sayısı: {len(ERA5_CITY_AREAS)}\n")

for city, coords in ERA5_CITY_AREAS.items():
    lat, lon = parse_coordinates_to_centroid(coords)
    
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue

    data = fetch_open_meteo_data(city, lat, lon)
    
    if data and "daily" in data:
        df = pd.DataFrame({
            "date": pd.to_datetime(data["daily"]["time"]),
            "uv_max": data["daily"]["uv_index_max"],
            "sun_sec": data["daily"]["sunshine_duration"]
        })
        
        # --- DÜZELTME BAŞLANGICI ---
        # Uyarıyı önlemek için verilerin sayısal olduğundan emin oluyoruz.
        # "coerce" hatalı/boş verileri NaN yapar.
        df["uv_max"] = pd.to_numeric(df["uv_max"], errors='coerce').fillna(0)
        df["sun_sec"] = pd.to_numeric(df["sun_sec"], errors='coerce').fillna(0)
        # --- DÜZELTME BİTİŞİ ---

        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # --- AYLIK HESAPLAMALAR ---
        # 'ME' (Month End) kullanıyoruz.
        monthly_uv = df.resample('ME', on='date')["uv_max"].mean()
        monthly_sun = df.resample('ME', on='date')["sun_hours"].sum()
        
        uv_row = {"Sehir": city}
        sun_row = {"Sehir": city}
        
        for date_idx, val in monthly_uv.items():
            col_name = date_idx.strftime("%Y-%m")
            uv_row[col_name] = round(val, 2)
            
        for date_idx, val in monthly_sun.items():
            col_name = date_idx.strftime("%Y-%m")
            sun_row[col_name] = round(val, 2)
            
        uv_data_list.append(uv_row)
        sun_data_list.append(sun_row)
        
    else:
        print(f"\n⚠️ Veri boş döndü: {city}")

    time.sleep(0.5)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if uv_data_list:
    df_uv_final = pd.DataFrame(uv_data_list)
    df_uv_final.to_excel(FILE_UV, index=False)
    print(f"✅ UV Dosyası Hazır: {FILE_UV}")

if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    print(f"✅ Güneşlenme Dosyası Hazır: {FILE_SUN}")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ İşlem Başlıyor... (2020-01-01 - 2022-12-31)
🌍 Toplam Şehir Sayısı: 119

📡 Veri çekiliyor (San Juan - Puerto Rico)...a)...s)....
❌ Hata (San Juan - Puerto Rico): 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=18.41&longitude=-65.96&start_date=2020-01-01&end_date=2022-12-31&daily=uv_index_max&daily=sunshine_duration&timezone=auto

⚠️ Veri boş döndü: San Juan - Puerto Rico
📡 Veri çekiliyor (Providence - Rhode Island)...
❌ Hata (Providence - Rhode Island): 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=41.67&longitude=-71.35&start_date=2020-01-01&end_date=2022-12-31&daily=uv_index_max&daily=sunshine_duration&timezone=auto

⚠️ Veri boş döndü: Providence - Rhode Island
📡 Veri çekiliyor (Charleston - South Carolina)...
❌ Hata (Charleston - South Carolina): 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=32.78&longitude=-79.93&start_dat

In [4]:
import requests
import pandas as pd

# --- AYARLAR ---
# Test edilecek lokasyon (Örn: Ankara)
LAT = 39.9334
LON = 32.8597

# Test edilecek tarih aralığı
START_DATE = "2020-01-01"
END_DATE = "2022-12-31"

# --- URL DİKKAT: "api.open-meteo" DEĞİL "archive-api.open-meteo" KULLANILIYOR ---
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": "uv_index_max", # UV Index verisini istiyoruz
    "timezone": "auto"
}

print(f"📡 {START_DATE} ile {END_DATE} arası UV Index verisi sorgulanıyor...")
print(f"🔗 İstek atılan adres: {url}")

try:
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        # Gelen veriyi kontrol et
        if "daily" in data and "uv_index_max" in data["daily"]:
            uv_values = data["daily"]["uv_index_max"]
            dates = data["daily"]["time"]
            
            # Pandas DataFrame oluştur
            df = pd.DataFrame({
                "Tarih": dates,
                "UV_Index": uv_values
            })
            
            # None (Boş) olmayan veri sayısını bul
            total_days = len(df)
            valid_days = df["UV_Index"].count() # NaN olmayanları sayar
            missing_days = total_days - valid_days
            
            print("\n✅ SONUÇ BAŞARILI: Veri Bulundu!")
            print("-" * 40)
            print(f"Toplam Gün Sayısı  : {total_days}")
            print(f"Dolu UV Verisi     : {valid_days}")
            print(f"Eksik/Boş Veri     : {missing_days}")
            print("-" * 40)
            
            # Yıllara göre dağılımı gösterelim ki emin olalım
            df["Yıl"] = pd.to_datetime(df["Tarih"]).dt.year
            print("\n📅 Yıllara Göre Dolu Veri Sayısı:")
            print(df.groupby("Yıl")["UV_Index"].count())
            
            print("\n🔎 Örnek Veriler (İlk 5 Gün):")
            print(df.head())
            
        else:
            print("❌ Yanıt geldi ama içinde 'uv_index_max' verisi yok.")
            print("Gelen Anahtarlar:", data.keys())
            
    else:
        print(f"❌ Hata Oluştu. Durum Kodu: {response.status_code}")
        print("Mesaj:", response.text)

except Exception as e:
    print(f"❌ Bağlantı Hatası: {e}")

📡 2020-01-01 ile 2022-12-31 arası UV Index verisi sorgulanıyor...
🔗 İstek atılan adres: https://archive-api.open-meteo.com/v1/archive

✅ SONUÇ BAŞARILI: Veri Bulundu!
----------------------------------------
Toplam Gün Sayısı  : 1096
Dolu UV Verisi     : 0
Eksik/Boş Veri     : 1096
----------------------------------------

📅 Yıllara Göre Dolu Veri Sayısı:
Yıl
2020    0
2021    0
2022    0
Name: UV_Index, dtype: int64

🔎 Örnek Veriler (İlk 5 Gün):
        Tarih UV_Index   Yıl
0  2020-01-01     None  2020
1  2020-01-02     None  2020
2  2020-01-03     None  2020
3  2020-01-04     None  2020
4  2020-01-05     None  2020


In [6]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
END_DATE = "2022-12-31"

# Çıktı Dosya İsmi (Rapor)
FILE_SUN = "Rapor_Aylik_Toplam_Guneslenme_Saati.xlsx"

# Ham verilerin saklanacağı klasör
CACHE_FOLDER = "sehir_verileri"

# Klasör yoksa oluştur
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    # Dosya isminde sorun çıkarabilecek karakterleri temizle
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["sunshine_duration"],
        "timezone": "auto"
    }
    print(f"📡 Veri indiriliyor ({city_name})...", end="\r")
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

sun_data_list = []

print(f"⏳ İşlem Başlıyor... ({START_DATE} - {END_DATE})")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne kaydedilecek.")
print(f"🌍 Toplam Şehir Sayısı: {len(ERA5_CITY_AREAS)}\n")

for city, coords in ERA5_CITY_AREAS.items():
    
    # 1. Koordinatları al
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue
    
    # 2. Dosya kontrolü (Varsa indirmez)
    file_path = get_safe_filename(city)
    df = None
    
    if os.path.exists(file_path):
        print(f"📁 Dosyadan okunuyor: {city}" + " " * 20, end="\r")
        try:
            df = pd.read_csv(file_path)
            # CSV'den okuyunca tarih string gelir, onu datetime'a çeviriyoruz
            df["date"] = pd.to_datetime(df["date"])
        except Exception as e:
            print(f"\n⚠️ Dosya okuma hatası ({city}): {e}")
            # Hata varsa aşağıda API'den tekrar çekmeyi deneyebiliriz, şimdilik geçiyoruz.
            continue
            
    else:
        # Dosya yoksa API'den çek
        data = fetch_open_meteo_data(city, lat, lon)
        
        if data and "daily" in data:
            df = pd.DataFrame({
                "date": pd.to_datetime(data["daily"]["time"]),
                "sun_sec": data["daily"]["sunshine_duration"]
            })
            
            # Veriyi temizle ve CSV'ye kaydet
            df["sun_sec"] = pd.to_numeric(df["sun_sec"], errors='coerce').fillna(0)
            df.to_csv(file_path, index=False) # <--- Klasöre kaydetme işlemi
            
            # API'den çektikten sonra bekleme (User isteği: 1 saniye)
            time.sleep(1)
        else:
            print(f"\n⚠️ API'den veri dönmedi: {city}")
            continue

    # 3. Hesaplama (Veri ister dosyadan gelsin ister API'den, burada işlenir)
    if df is not None:
        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # Aylık Toplam Güneşlenme (Sum) - 'ME' = Month End
        monthly_sun = df.resample('ME', on='date')["sun_hours"].sum()
        
        sun_row = {"Sehir": city}
        
        for date_idx, val in monthly_sun.items():
            col_name = date_idx.strftime("%Y-%m")
            sun_row[col_name] = round(val, 2)
            
        sun_data_list.append(sun_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    print(f"✅ Güneşlenme Dosyası Hazır: {FILE_SUN}")
else:
    print("⚠️ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ İşlem Başlıyor... (2020-01-01 - 2022-12-31)
📂 Veriler 'sehir_verileri' klasörüne kaydedilecek.
🌍 Toplam Şehir Sayısı: 119

📁 Dosyadan okunuyor: Abu Dhabi - United Arab Emirates                                

💾 Veriler işleniyor ve Excel'e kaydediliyor...
✅ Güneşlenme Dosyası Hazır: Rapor_Aylik_Toplam_Guneslenme_Saati.xlsx

🏁 Tüm işlemler tamamlandı.


In [3]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
# 🌟 Hedef Bitiş Tarihi: 15 Ocak 2023
END_DATE = "2023-01-15" 

# Çıktı Dosya İsmi (Rapor)
FILE_SUN = "Rapor_Aylik_Toplam_Guneslenme_Saati_37Ay.xlsx"

# Ham verilerin saklanacağı klasör
CACHE_FOLDER = "sehir_verileri_sun_37ay"

# Klasör yoksa oluştur
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi (Aynı liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    # Dosya isminde sorun çıkarabilecek karakterleri temizle
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE, # 🌟 2023-01-15'e kadar
        "daily": ["sunshine_duration"], 
        "timezone": "auto"
    }
    print(f"📡 Veri indiriliyor ({city_name})...", end="\r")
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

sun_data_list = []

print(f"⏳ Güneşlenme Süresi İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print("ℹ️  Veri seti 37 ayı (Ocak 2020 - 15 Ocak 2023) kapsayacak.\n")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne kaydedilecek.")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # 2. Dosya Kontrolü
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except Exception as e:
            print(f"\n⚠️ Dosya okuma hatası ({city}): {e}")
            continue
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_open_meteo_data(city, lat, lon)
        
        if data and "daily" in data:
            df = pd.DataFrame({
                "date": pd.to_datetime(data["daily"]["time"]),
                "sun_sec": data["daily"]["sunshine_duration"]
            })
            
            # Veriyi Temizle ve Kaydet
            df["sun_sec"] = pd.to_numeric(df["sun_sec"], errors='coerce').fillna(0)
            df.to_csv(file_path, index=False) 
            
            time.sleep(1) # Nezaket beklemesi
        else:
            print(f"\n⚠️ API'den veri dönmedi: {city}")
            continue

    # 3. Hesaplama 
    if df is not None:
        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # Aylık Toplam Güneşlenme (SUM)
        # 2023-01 için sadece 15 günlük veri var, bunların toplamını alacak.
        monthly_sun = df.resample('ME', on='date')["sun_hours"].sum()
        
        sun_row = {"Sehir": city}
        
        for date_idx, val in monthly_sun.items():
            col_name = date_idx.strftime("%Y-%m")
            sun_row[col_name] = round(val, 2)
            
        sun_data_list.append(sun_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    
    print(f"✅ Güneşlenme Dosyası Hazır: {FILE_SUN}")
    # Sütun sayısını kontrol et (Şehir hariç 37 olmalı)
    print(f"ℹ️  Toplam Ay Sayısı: {df_sun_final.shape[1] - 1}") 
else:
    print("⚠️ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ Güneşlenme Süresi İşlemi Başlıyor... (2020-01-01 - 2023-01-15)
ℹ️  Veri seti 37 ayı (Ocak 2020 - 15 Ocak 2023) kapsayacak.

📂 Veriler 'sehir_verileri_sun_37ay' klasörüne kaydedilecek.
📡 Veri indiriliyor (San Marino)...

💾 Veriler işleniyor ve Excel'e kaydediliyor...
✅ Güneşlenme Dosyası Hazır: Rapor_Aylik_Toplam_Guneslenme_Saati_37Ay.xlsx
ℹ️  Toplam Ay Sayısı: 37

🏁 Tüm işlemler tamamlandı.


In [ ]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
START_DATE = "1990-01-01"
END_DATE = "1993-01-15" 

# Çıktı Dosya İsmi (Günlük Rapor)
FILE_SUN = "Rapor_Gunluk_Guneslenme_Saati_1111Gun_old.xlsx"

# Ham verilerin saklanacağı klasör
CACHE_FOLDER = "sehir_verileri_sun_gunluk_old"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi (Aynı liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["sunshine_duration"], 
        "timezone": "auto"
    }
    print(f"📡 Veri indiriliyor ({city_name})...", end="\r")
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

# Verileri sütun bazlı (dictionary key = Şehir İsmi) saklayacağız,
# çünkü Pandas dictionary'den DataFrame yaparken en hızlı böyle çalışır.
# Ancak siz Şehirleri SATIRDA, Günleri SÜTUNDA istediğiniz için
# en sonda Transpose (.T) alacağız.
all_cities_data = {} 

print(f"⏳ Günlük Güneşlenme Verisi İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne önbelleğe alınacak.")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # 2. Dosya Kontrolü
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except Exception as e:
            print(f"\n⚠️ Dosya okuma hatası ({city}): {e}")
            continue
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_open_meteo_data(city, lat, lon)
        
        if data and "daily" in data:
            df = pd.DataFrame({
                "date": pd.to_datetime(data["daily"]["time"]),
                "sun_sec": data["daily"]["sunshine_duration"]
            })
            
            # Veriyi Temizle ve Kaydet
            df["sun_sec"] = pd.to_numeric(df["sun_sec"], errors='coerce').fillna(0)
            df.to_csv(file_path, index=False) 
            
            time.sleep(1) # Nezaket beklemesi
        else:
            print(f"\n⚠️ API'den veri dönmedi: {city}")
            continue

    # 3. Hesaplama ve Dictionary'e Atma
    if df is not None:
        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        df["sun_hours"] = df["sun_hours"].round(2) # 2 ondalık basamak
        
        # DataFrame'i 'date' indexli hale getirip sadece sun_hours serisini alıyoruz
        # Bu sayede tarihler otomatik eşleşecek.
        series = df.set_index("date")["sun_hours"]
        
        # Bu seriyi ana sözlüğe ekle (Key: Şehir Adı)
        all_cities_data[city] = series

print("\n\n💾 Veriler birleştiriliyor ve Excel'e kaydediliyor...")

if all_cities_data:
    # 1. Dictionary'den DataFrame oluştur (Sütunlar: Şehirler, Satırlar: Tarihler)
    df_temp = pd.DataFrame(all_cities_data)
    
    # 2. Transpose al (Satırlar: Şehirler, Sütunlar: Tarihler)
    df_final = df_temp.T
    
    # 3. Sütun isimlerini (Tarihleri) string formatına çevir ki Excel'de düzgün görünsün
    # (YYYY-MM-DD formatı)
    df_final.columns = [col.strftime('%Y-%m-%d') for col in df_final.columns]
    
    # 4. Şehir ismini index'ten çıkarıp normal sütun yap (Opsiyonel, daha temiz görünür)
    df_final.reset_index(inplace=True)
    df_final.rename(columns={'index': 'Sehir'}, inplace=True)

    # 5. Kaydet
    df_final.to_excel(FILE_SUN, index=False)
    
    print(f"✅ Güneşlenme Dosyası Hazır: {FILE_SUN}")
    print(f"ℹ️  Toplam Gün (Sütun) Sayısı: {df_final.shape[1] - 1}") # Şehir sütunu hariç
else:
    print("⚠️ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ Günlük Güneşlenme Verisi İşlemi Başlıyor... (1990-01-01 - 1993-01-15)
📂 Veriler 'sehir_verileri_sun_gunluk_old' klasörüne önbelleğe alınacak.


In [6]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
START_DATE = "2020-01-01"
# 🌟 Hedef Bitiş Tarihi: 15 Ocak 2023
END_DATE = "2023-01-15" 

# Çıktı Dosya İsmi (Günlük Rapor)
FILE_SUN = "Rapor_Gunluk_Guneslenme_Saati_37Ay.xlsx"

# Ham verilerin saklanacağı klasör
CACHE_FOLDER = "sehir_verileri_sun_37ay"

# Klasör yoksa oluştur
os.makedirs(CACHE_FOLDER, exist_ok=True)

# 🌍 ŞEHİR LİSTESİ (Aynı Koordinatlar)
ERA5_CITY_AREAS = {
    # --- ABD Eyalet Başkentleri ve Bölgeler ---
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa - Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Frankfort - Kentucky': '38.4500/-85.1200/37.9500/-84.6200',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.5600/-69.9600/44.0600/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Saint Paul - Minnesota': '45.1900/-93.3500/44.6900/-92.8500',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston AFB - South Carolina': '33.1400/-80.2900/32.6400/-79.7900',
    'Columbia - South Dakota': '45.8500/-98.5500/45.3500/-98.0500',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'Virgin Islands': '18.5800/-65.1500/18.0800/-64.6500',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # --- Uluslararası Şehirler ve Bölgeler ---
    'Tirana - Albania': '41.5700/19.5600/41.0700/20.0600',
    'Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3700/-62.0800/16.8700/-61.5800',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Vienna - Austria': '48.4500/16.1200/47.9500/16.6200',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Sarajevo - Bosnia and Herzegovina': '44.1000/18.1600/43.6000/18.6600',
    'Road Town - British Virgin Islands': '18.6700/-64.8700/18.1700/-64.3700',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Zagreb - Croatia': '46.0600/15.7300/45.5600/16.2300',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prague - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.5500/-61.6300/15.0500/-61.1300',
    'San Salvador - El Salvador': '13.9400/-89.4700/13.4400/-88.9700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s - Grenada': '12.3000/-61.9500/11.8000/-61.4500',
    'Port-au-Prince - Haiti': '18.7900/-72.5800/18.2900/-72.0800',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Budapest - Hungary': '47.7400/18.7900/47.2400/19.2900',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.9100/20.9100/42.4100/21.4100',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Skopje - North Macedonia': '42.2500/21.1800/41.7500/21.6800',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.5500/-62.9700/17.0500/-62.4700',
    'Castries - Saint Lucia': '14.2600/-61.2400/13.7600/-60.7400',
    'Kingstown - Saint Vincent and the Grenadines': '13.4100/-61.4700/12.9100/-60.9700',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Belgrade - Serbia': '45.0300/20.1900/44.5300/20.6900',
    'Victoria - Seychelles': '-4.3700/55.2000/-4.8700/55.7000',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200',
    'Montevideo - Uruguay': '-34.6500/-56.4100/-35.1500/-55.9100'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    # Dosya isminde sorun çıkarabilecek karakterleri temizle
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_open_meteo_data(city_name, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE, 
        "daily": ["sunshine_duration"], 
        "timezone": "auto"
    }
    print(f"📡 Veri indiriliyor ({city_name})...", end="\r")
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

sun_data_list = []

print(f"⏳ Günlük Güneşlenme Süresi İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne kaydedilecek.")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # 2. Dosya Kontrolü
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except Exception as e:
            print(f"\n⚠️ Dosya okuma hatası ({city}): {e}")
            continue
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_open_meteo_data(city, lat, lon)
        
        if data and "daily" in data:
            df = pd.DataFrame({
                "date": pd.to_datetime(data["daily"]["time"]),
                "sun_sec": data["daily"]["sunshine_duration"]
            })
            
            # Veriyi Temizle ve Kaydet
            df["sun_sec"] = pd.to_numeric(df["sun_sec"], errors='coerce').fillna(0)
            df.to_csv(file_path, index=False) 
            
            time.sleep(1) # Nezaket beklemesi
        else:
            print(f"\n⚠️ API'den veri dönmedi: {city}")
            continue

    # 3. Hesaplama (GÜNLÜK VERİLER İÇİN)
    if df is not None:
        # Saniyeyi saate çevir
        df["sun_hours"] = df["sun_sec"] / 3600
        
        # Tarih formatını string yap (Sütun isimleri için)
        df['date_str'] = df['date'].dt.strftime('%Y-%m-%d')
        
        # Şehir adını ekle
        sun_row = {"Sehir": city}
        
        # Tüm günleri sözlüğe ekle (Resample yapmadan doğrudan aktarıyoruz)
        # Bu yöntem döngüden çok daha hızlıdır
        daily_data = df.set_index('date_str')['sun_hours'].to_dict()
        
        # Virgülden sonra 2 basamak yuvarlayalım
        daily_data = {k: round(v, 2) for k, v in daily_data.items()}
        
        sun_row.update(daily_data)
        sun_data_list.append(sun_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if sun_data_list:
    df_sun_final = pd.DataFrame(sun_data_list)
    df_sun_final.to_excel(FILE_SUN, index=False)
    
    print(f"✅ GÜNLÜK Güneşlenme Dosyası Hazır: {FILE_SUN}")
    # Sütun sayısını kontrol et
    print(f"ℹ️  Toplam Gün Sayısı (Sütun): {df_sun_final.shape[1] - 1}") 
else:
    print("⚠️ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ Günlük Güneşlenme Süresi İşlemi Başlıyor... (2020-01-01 - 2023-01-15)
📂 Veriler 'sehir_verileri_sun_37ay' klasörüne kaydedilecek.
📡 Veri indiriliyor (Montevideo - Uruguay)...Emirates)......dines)...

💾 Veriler işleniyor ve Excel'e kaydediliyor...
✅ GÜNLÜK Güneşlenme Dosyası Hazır: Rapor_Gunluk_Guneslenme_Saati_37Ay.xlsx
ℹ️  Toplam Gün Sayısı (Sütun): 1111

🏁 Tüm işlemler tamamlandı.
